# Week 10 Hands-On Lab — Probing Three Trustworthy-AI Pillars

**ESP3201 · formative hands-on lab.** Runs on free-tier Google Colab with a **CPU runtime** (no GPU, no API keys).

A "trustworthy AI" report that passes six pillar checks is a claim, not a proof. This lab gives you
real, runnable evidence for three of the pillars -- fairness, explainability, and privacy -- on toy
data you generate and inspect yourself. The graded Week 10 mini-assignment then asks you to apply
the same recomputation skill to a real (if fictional) audit report and its own data.

**Note:** the data in this lab is a separate illustrative example, not the data used in the graded
mini-assignment.

## Tasks

1. Recompute a fairness headline number from raw per-cell data and find the subgroup it hides.
2. Compare a model's local explanation at one instance against its actual global behavior.
3. Measure a real membership-inference confidence gap on a small, overfit model.
4. Fill the worksheet.

## Setup

In [ ]:
import os
try:
    import sklearn
except ModuleNotFoundError:
    os.system("pip install -q scikit-learn")

import numpy as np

# --- Week 10 lab core, embedded directly (no repo clone) ----------------------
# Canonical source, kept in sync by hand: starter/trustworthy_ai_lab.py in the
# course repo. Cloning a support module from Colab is fragile -- if a session
# already ran once before an update landed, the clone silently no-ops onto the
# stale cached copy instead of fetching the fix. Pasting the needed code
# directly here removes that whole failure class.

import csv
import io
from dataclasses import dataclass

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier


# ---------------------------------------------------------------------------
# Pillar 1: Fairness -- recompute a headline number from raw per-cell data.
# ---------------------------------------------------------------------------

def load_subgroup_csv(path_or_text: str) -> list[dict]:
    """Load a fairness subgroup CSV (condition, lighting, subgroup, n_samples,
    detections_correct) from a file path or raw CSV text."""
    text = path_or_text
    try:
        with open(path_or_text, encoding="utf-8") as f:
            text = f.read()
    except (FileNotFoundError, OSError):
        pass
    reader = csv.DictReader(io.StringIO(text))
    rows = []
    for row in reader:
        row["n_samples"] = int(row["n_samples"])
        row["detections_correct"] = int(row["detections_correct"])
        rows.append(row)
    return rows


def weighted_accuracy(rows: list[dict]) -> float:
    """The single headline number a report would quote: total correct / total samples."""
    total_n = sum(r["n_samples"] for r in rows)
    total_correct = sum(r["detections_correct"] for r in rows)
    if total_n == 0:
        raise ValueError("no samples")
    return total_correct / total_n


def per_cell_accuracy(rows: list[dict]) -> list[dict]:
    """Per-cell accuracy -- the numbers a headline accuracy can hide."""
    out = []
    for r in rows:
        acc = r["detections_correct"] / r["n_samples"] if r["n_samples"] else 0.0
        out.append({**r, "accuracy": acc})
    return out


def worst_cell(rows: list[dict]) -> dict:
    """The cell a weighted-mean headline is most likely to be hiding."""
    cells = per_cell_accuracy(rows)
    return min(cells, key=lambda r: r["accuracy"])


# ---------------------------------------------------------------------------
# Pillar 2: Explainability -- a local explanation is not proof of global reasoning.
# ---------------------------------------------------------------------------

def make_explainability_demo_data(seed: int = 0, n: int = 400):
    """Two-feature synthetic data. The TRUE label depends only on feature A (signal).
    Feature B is pure noise but happens to correlate with the label for a specific,
    narrow region of A -- exactly the kind of local pattern a post-hoc explanation
    can mistake for the model's real reasoning."""
    rng = np.random.default_rng(seed)
    a = rng.uniform(-3, 3, n)
    b = rng.normal(0, 1, n)
    y = (a > 0).astype(int)
    # In a narrow band of `a`, make `b` spuriously predictive by construction --
    # this is the region a local explanation is likely to be probed on.
    band = (a > -1.2) & (a < 1.2)
    b[band] = np.where(y[band] == 1, b[band] + 7.0, b[band] - 7.0)
    X = np.column_stack([a, b])
    return X, y


def train_explainability_demo_model(X, y):
    model = LogisticRegression()
    model.fit(X, y)
    return model


def local_explanation(model, x_row: np.ndarray, X_background: np.ndarray, n_samples: int = 200, seed: int = 0) -> dict:
    """A minimal, from-scratch local explanation: perturb one feature at a time
    (holding the others at the instance's own value) and measure how much the
    model's predicted probability moves. This is the same idea LIME/SHAP formalize --
    a *local* sensitivity measurement, not a description of what the model does
    everywhere."""
    rng = np.random.default_rng(seed)
    base_prob = model.predict_proba(x_row.reshape(1, -1))[0, 1]
    importances = {}
    for feat_idx, feat_name in enumerate(["feature_a", "feature_b"]):
        perturbed = np.tile(x_row, (n_samples, 1))
        perturbed[:, feat_idx] = rng.choice(X_background[:, feat_idx], size=n_samples, replace=True)
        probs = model.predict_proba(perturbed)[:, 1]
        importances[feat_name] = float(np.mean(np.abs(probs - base_prob)))
    return {"base_prob": float(base_prob), "local_importance": importances}


def global_feature_weight(model) -> dict:
    """The model's actual (global) linear coefficients, for comparison against the
    local explanation above."""
    coef = model.coef_[0]
    return {"feature_a": float(coef[0]), "feature_b": float(coef[1])}


# ---------------------------------------------------------------------------
# Pillar 3: Privacy -- a trained model can leak whether a point was in training data.
# ---------------------------------------------------------------------------

def make_membership_inference_demo_data(seed: int = 0, n_train: int = 40, n_test: int = 300):
    """A small, high-capacity model trained on very few, noisy points is the classic
    setup for a real (if toy) membership-inference gap: it memorizes its small
    training set more confidently than it generalizes to held-out points."""
    rng = np.random.default_rng(seed)
    n_features = 25
    X_train = rng.normal(0, 1, (n_train, n_features))
    y_train = (X_train[:, 0] + 1.5 * rng.normal(0, 1, n_train) > 0).astype(int)
    X_test = rng.normal(0, 1, (n_test, n_features))
    y_test = (X_test[:, 0] + 1.5 * rng.normal(0, 1, n_test) > 0).astype(int)
    return X_train, y_train, X_test, y_test


def train_membership_inference_demo_model(X_train, y_train):
    # Deliberately high-capacity relative to n_train=40, unregularized (alpha=0) --
    # this is what makes the model memorize rather than generalize, which is the
    # whole point of the demo, not a mistake to "fix".
    model = MLPClassifier(hidden_layer_sizes=(200, 200), max_iter=3000, alpha=0.0, random_state=0)
    model.fit(X_train, y_train)
    return model


def membership_confidence_gap(model, X_train, X_test) -> dict:
    """Mean predicted-probability confidence on training points vs. held-out points.
    A real membership-inference attack does exactly this: query a model twice on
    a point-you-suspect-was-training-data and a point-you-know-was-not, and use the
    confidence gap to guess which set it came from."""
    train_conf = model.predict_proba(X_train).max(axis=1)
    test_conf = model.predict_proba(X_test).max(axis=1)
    return {
        "mean_train_confidence": float(train_conf.mean()),
        "mean_test_confidence": float(test_conf.mean()),
        "gap": float(train_conf.mean() - test_conf.mean()),
    }

# The fairness CSV, embedded the same way as the rest of the lab core above --
# also the canonical source at starter/fairness_probe_data.csv.
FAIRNESS_CSV = """condition,lighting,subgroup,n_samples,detections_correct
indoor,bright,adult,800,784
indoor,bright,child,300,282
indoor,dim,adult,500,470
indoor,dim,child,200,164
outdoor,bright,adult,700,686
outdoor,bright,child,250,235
outdoor,dim,adult,450,414
outdoor,dim,child,150,111
"""

print("trustworthy_ai_lab loaded (embedded in this notebook -- no repo clone needed).")

## Part A — Fairness: recompute the headline, find the hidden cell

A report can quote one weighted-average accuracy number and never show the per-condition breakdown.
Below is `fairness_probe_data.csv`: detection results for a toy person-detector across
indoor/outdoor and bright/dim conditions, split by an `adult`/`child` subgroup.

In [ ]:
rows = load_subgroup_csv(FAIRNESS_CSV)
for r in rows:
    print(r)

**First**, compute the single weighted-average accuracy a report would quote (this is exactly
`total_correct / total_samples` across every row -- write it yourself before running the next cell).

In [ ]:
headline = weighted_accuracy(rows)
print(f"weighted overall accuracy: {headline:.4f}")

**Now** break it down by cell, and find the worst one. Does the headline number tell you this
cell exists?

In [ ]:
for cell in per_cell_accuracy(rows):
    print(f"{cell['condition']:8s} {cell['lighting']:6s} {cell['subgroup']:6s} "
          f"n={cell['n_samples']:4d}  accuracy={cell['accuracy']:.3f}")

print()
print("WORST CELL:", worst_cell(rows))

### See it, don't just read it

Eight per-cell accuracy numbers in a printed table are easy to skim past. Below: the same cells
as a bar chart, with the headline drawn as a reference line -- the gap between the worst bar and
that line is the whole point of this pillar.

In [ ]:
import matplotlib.pyplot as plt

cells = per_cell_accuracy(rows)
fig, ax = plt.subplots(figsize=(8, 4.2))
labels = [f"{c['condition']}\n{c['lighting']}/{c['subgroup']}" for c in cells]
accs = [c["accuracy"] for c in cells]
colors = ["#c44e52" if c["subgroup"] == "child" else "#4c72b0" for c in cells]
ax.bar(range(len(cells)), accs, color=colors)
ax.axhline(headline, color="black", linestyle="--", linewidth=1.5, label=f"headline = {headline:.3f}")
ax.set_xticks(range(len(cells)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylim(0, 1.12)
ax.set_ylabel("accuracy")
ax.set_title("Per-cell accuracy vs. the headline (red = child subgroup)")
ax.legend(fontsize=9, loc="lower left")
for i, c in enumerate(cells):
    ax.text(i, c["accuracy"] + 0.03, f"n={c['n_samples']}", ha="center", fontsize=7)
plt.tight_layout()
plt.show()

### Checkpoint A — what does the headline hide?

In your own words: how far below the headline accuracy is the worst cell, and is that gap large enough that you'd want to know about it before deploying this detector outdoors, in the dark, on children? What is the headline number actively *not telling you*?

## Part B — Explainability: a local explanation is not the whole model

We'll train a small classifier where the label truly depends on `feature_a`. `feature_b` is mostly
noise -- **except** in a narrow band of `feature_a` values, where it happens to correlate with the
label by construction. This mimics a real failure mode: a feature that is spuriously predictive in
one region of the data, which a *local* explanation method can mistake for the model's real
reasoning.

In [ ]:
X, y = make_explainability_demo_data(seed=0)
model = train_explainability_demo_model(X, y)

global_weights = global_feature_weight(model)
print("GLOBAL model weights (what the model actually relies on, on average):")
print(f"  feature_a: {global_weights['feature_a']:.3f}")
print(f"  feature_b: {global_weights['feature_b']:.3f}")

Now pick an instance right in the middle of that narrow band (where `feature_a` is close to its decision boundary) and generate a **local** explanation for it.

In [ ]:
idx = int(np.argmin(np.abs(X[:, 0])))
x_row = X[idx]
print(f"instance: feature_a={x_row[0]:.3f}  feature_b={x_row[1]:.3f}  true label={y[idx]}")

explanation = local_explanation(model, x_row, X)
print("model\'s predicted probability of class 1 at this instance:", explanation["base_prob"])
print("LOCAL importance at this instance:")
print(f"  feature_a: {explanation['local_importance']['feature_a']:.3f}")
print(f"  feature_b: {explanation['local_importance']['feature_b']:.3f}")

### See why they disagree, not just that they do

The two printed numbers above (global weight vs. local importance) tell you *that* they disagree.
This plot shows *why*: the probed instance sits inside the narrow band where `feature_b` was
deliberately made spuriously predictive, so a local probe centered there picks up that local
correlation -- even though it explains almost nothing about the model's behavior everywhere
else.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for label_val, color, name in [(0, "#4c72b0", "label=0"), (1, "#c44e52", "label=1")]:
    mask = y == label_val
    ax.scatter(X[mask, 0], X[mask, 1], s=14, alpha=0.55, color=color, label=name)
ax.axvspan(-1.2, 1.2, color="gold", alpha=0.2, label="spurious band (-1.2 < feature_a < 1.2)")
ax.scatter([x_row[0]], [x_row[1]], s=200, marker="*", color="black", zorder=5, label="probed instance")
ax.set_xlabel("feature_a (the TRUE signal)")
ax.set_ylabel("feature_b (noise, except inside the band)")
ax.set_title("Why local and global explanations disagree")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

### Checkpoint B — local versus global

Which feature does the GLOBAL weight say the model relies on? Which feature does the LOCAL explanation say mattered most *at this one instance*? If these disagree, what would a stakeholder wrongly conclude from reading only the local explanation for this instance? Is the local explanation *wrong*, or is it correct but insufficient as a claim about the model overall?

## Part C — Privacy: can you tell if a point was in the training set?

Train a small, high-capacity, unregularized model on a **very small** training set. A real
membership-inference attack asks: given a model and a data point, can you tell whether that point
was used to train it? One simple signal: the model's own confidence.

In [ ]:
X_train, y_train, X_test, y_test = make_membership_inference_demo_data(seed=0)
model = train_membership_inference_demo_model(X_train, y_train)

print(f"train accuracy: {model.score(X_train, y_train):.3f}")
print(f"test accuracy:  {model.score(X_test, y_test):.3f}")

gap = membership_confidence_gap(model, X_train, X_test)
print()
print(f"mean confidence on TRAINING points: {gap['mean_train_confidence']:.4f}")
print(f"mean confidence on HELD-OUT points: {gap['mean_test_confidence']:.4f}")
print(f"confidence gap:                    {gap['gap']:.4f}")

### See the two distributions, not just their two means

`gap` above is one number: the difference between two means. The actual membership-inference
attack works on the full distributions -- if you had to guess "was this single point in the
training set?" using only its confidence, these are the two distributions you'd be
distinguishing.

In [ ]:
import numpy as np

train_conf = model.predict_proba(X_train).max(axis=1)
test_conf = model.predict_proba(X_test).max(axis=1)

fig, ax = plt.subplots(figsize=(6.5, 4))
bins = np.linspace(0.4, 1.0, 25)
ax.hist(train_conf, bins=bins, alpha=0.6, color="#c44e52", density=True,
        label=f"training points (n={len(train_conf)}, mean={train_conf.mean():.3f})")
ax.hist(test_conf, bins=bins, alpha=0.6, color="#4c72b0", density=True,
        label=f"held-out points (n={len(test_conf)}, mean={test_conf.mean():.3f})")
ax.axvline(train_conf.mean(), color="#c44e52", linestyle="--", linewidth=1.5)
ax.axvline(test_conf.mean(), color="#4c72b0", linestyle="--", linewidth=1.5)
ax.set_xlabel("model confidence (max predicted probability)")
ax.set_ylabel("density (normalized -- sample sizes differ)")
ax.set_title(f"Membership-inference signal: confidence gap = {gap['gap']:.3f}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Zoomed out: does the leak go away with a smaller model?

Sweep the hidden-layer size (holding `n_train=40` fixed) and track both the confidence gap and
training accuracy. **Don't assume the answer -- the cell below runs it.**

In [ ]:
from sklearn.neural_network import MLPClassifier

hidden_sizes = [2, 4, 8, 16, 32, 64, 128]
gaps, train_accs, test_accs = [], [], []
for h in hidden_sizes:
    sweep_model = MLPClassifier(hidden_layer_sizes=(h,), max_iter=3000, alpha=0.0, random_state=0)
    sweep_model.fit(X_train, y_train)
    tr_conf = sweep_model.predict_proba(X_train).max(axis=1)
    te_conf = sweep_model.predict_proba(X_test).max(axis=1)
    gaps.append(tr_conf.mean() - te_conf.mean())
    train_accs.append(sweep_model.score(X_train, y_train))
    test_accs.append(sweep_model.score(X_test, y_test))

fig, ax1 = plt.subplots(figsize=(6.5, 4.2))
ax1.plot(hidden_sizes, gaps, "o-", color="#8172b2", label="confidence gap (privacy leak)")
ax1.set_xscale("log", base=2)
ax1.set_xticks(hidden_sizes); ax1.set_xticklabels(hidden_sizes)
ax1.set_xlabel("hidden layer size (model capacity)")
ax1.set_ylabel("confidence gap (train - test)", color="#8172b2")
ax1.tick_params(axis="y", labelcolor="#8172b2")

ax2 = ax1.twinx()
ax2.plot(hidden_sizes, train_accs, "s--", color="#c44e52", label="train accuracy (memorization)")
ax2.set_ylabel("train accuracy", color="#c44e52")
ax2.tick_params(axis="y", labelcolor="#c44e52")

fig.suptitle(f"Past perfect memorization (train_acc=1.0), more capacity only adds leak\n"
             f"(test accuracy stays flat around {min(test_accs):.2f}-{max(test_accs):.2f} throughout)",
             fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

**Read the plot, don't assume it.** Once the model is big enough to reach `train_acc=1.0`
(around `hidden=4` here), it has already memorized every training point -- growing it further
adds no *useful* accuracy (test accuracy stays flat), yet the privacy leak keeps climbing. The
leak past that point is pure unnecessary capacity, not a cost of achieving good accuracy. Only
the smallest model (`hidden=2`, which never reaches `train_acc=1.0`) shows a materially smaller
gap -- because it cannot memorize its training set well enough to *have* a memorization signal
to leak, not because it generalizes better.

### Checkpoint C — what does the gap prove?

Compare the train/test *accuracy* gap with the train/test *confidence* gap. If someone handed you a single data point and asked "was this in the training set?", could you make a real guess using only the model's confidence? What does this imply about publishing a model (even without publishing its training data)?

## Practical mitigations, beyond this toy demo

Each pillar here uses a deliberately small, controllable stand-in so the failure mode is cheap to
see. The mechanisms that actually address each one in production don't have a single on/off
switch:

- **Fairness.** The 8-cell breakdown here is by two known attributes (lighting, subgroup). Real
  deployments often don't know the sensitive subgroup at inference time, so the practical version
  is auditing on held-out slices *before* deployment, and monitoring per-slice metrics
  post-deployment rather than trusting one aggregate number to stay representative forever.
- **Explainability.** A local explanation (LIME/SHAP-style) is a *sample*, not a proof. Practical
  mitigations: check several local explanations across the input space before trusting any one of
  them, compare against the model's actual global feature importances (as done above), and treat
  a single local explanation presented to a stakeholder as a claim that needs corroboration, not
  a verdict.
- **Privacy.** The sweep above shows shrinking the model isn't a clean dial -- and in real
  settings you don't control test-time attacker access the way this toy demo does. Established
  mitigations: differential privacy during training (DP-SGD -- clips and noises gradients, with a
  formal privacy budget), regularization strong enough to change effective capacity (not just any
  nonzero `alpha`, as the sweep above shows), and simply not training on data you can't afford to
  leak signal about.

None of these make the failure mode disappear for free -- each trades accuracy, cost, or
engineering complexity for reduced risk, which is exactly why "just turn it off" (delete the
proxy, shrink the model, hide the sensitive attribute) is rarely the real fix.

## Your deliverable — a max-5-page PDF report

**Submit one PDF, 5 pages maximum.** The cap is hard — it includes every table, chart, and screenshot. **Condense:** a single chart or diagram that *compares* your conditions beats a long table or a wall of screenshots. Report only numbers your own run produced. Work the worksheet below into that report, and finish with the required **"How to improve this assignment"** section (last section).

## Worksheet (your deliverable)

### 1. Fairness

- Headline (weighted) accuracy:
- Worst cell (name it) and its accuracy:
- Gap between headline and worst cell:

### 2. Explainability

- Global weight ranking (which feature matters more overall):
- Local importance ranking at your chosen instance:
- Do they agree? If not, what is the risk of trusting only the local explanation?

### 3. Privacy

- Train accuracy / test accuracy:
- Train confidence / test confidence / gap:
- Could you use this gap to guess membership on a new point? Explain briefly.

### 4. Connect to the six pillars

Pick ONE of fairness, explainability, or privacy above and state, in one sentence, the evaluation
question this lab shows you cannot answer by reading a report's conclusion alone -- only by running
the numbers yourself.

## How to improve this assignment (required, ungraded)

*Required for a complete submission; it carries no marks.* In 3–5 sentences: what was unclear, too easy, too hard, or missing here? Name the **one change** that would make this a better learning exercise or a fairer test of the skill — a different probe, a harder map / scene / model, a metric that would have caught something this one missed, or a clearer instruction. Be specific; "it was fine" is not useful feedback.

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking